# Image-to-ASCII: model training and development

This notebook records development of the image-to-ASCII model: a 20×20 random
forest baseline followed by an incremental 10×10 SGD classifier. The local
webpage uses the saved SGD model and the supplied 10×10 character images.

Configure the paths in the first code cell and provide the input images before
running the relevant experiments. The character templates are included in
`assets/ascii_images` and `assets/ascii_images_shrunk`. Their optional generation
steps are disabled by default. Training and model export are optional and can
overwrite the supplied model when run.

Personal paths and cached outputs have been removed. Comments describe each
processing stage. Portability fixes cover string quoting, dynamic file counts,
and saving the final incomplete feature chunk. Some original vectorization and
character-cache code was developed with ChatGPT assistance; the local demo and
this edited presentation also received ChatGPT assistance.

Experimental details retained from development:

- SGD training uses edge detection; final inference uses greyscale without it.
- The baseline label-distance calculation uses uint8 arithmetic, which can overflow.
- The SGD train/test split is by feature chunk, so source images can overlap.
- Evaluation measures agreement with automatically assigned character labels,
  rather than visual quality or generalization to independent source images.


In [ ]:
# Find the project folder when running from the root or notebooks directory.
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Change these paths to suit the dataset and examples being used.
DATA_DIR = PROJECT_ROOT / "data"
ASSET_DIR = PROJECT_ROOT / "assets"
MODEL_DIR = PROJECT_ROOT / "models"
EXAMPLE_IMAGE = DATA_DIR / "test_images" / "example.jpg"

# Character PNGs are included, so no font installation is needed to use them.
# Only change these settings when deliberately rebuilding the character images.
REGENERATE_CHARACTERS = False
FONT_PATH = Path("path/to/segoeui.ttf")

# Create the folders used for input images and generated training data.
for name in ("photos_no_class", "photos2", "modified_images", "modified_images2",
             "image_bits", "image_bits2", "saved_arrays", "test_images"):
    (DATA_DIR / name).mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 1. Generate character templates

In [ ]:
# Draw one ASCII character onto a centred 20x20 greyscale image.
# This optional function recreates the original character templates.
# Set FONT_PATH to the intended font before enabling regeneration.
import math
from PIL import Image, ImageDraw, ImageFont

def ascii_images(ascii_code, size):
    font = ImageFont.truetype(
        FONT_PATH,
        size= size
    )
    text = chr(ascii_code)

    dummy_img = Image.new("L", (1, 1))
    dummy_draw = ImageDraw.Draw(dummy_img)
    # Measure the character so it can be centred within the tile.
    bbox = dummy_draw.textbbox((0,0), text, font=font)

    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    x = max(w, h)
    img = Image.new("L", (size, size))
    draw = ImageDraw.Draw(img)
    origin = -bbox[0],-bbox[1]

    # Offset the text bounds to place the character in the middle.
    adjustment = (size - w)//2, (size-h)//2
    adjusted_origin = origin[0] + adjustment[0], origin[1] + adjustment[1]
    draw.text((adjusted_origin), text, font=font, fill="white")

    filename = f"{ascii_code}.png"

    img.save(ASSET_DIR / "ascii_images" / filename, "PNG")


In [ ]:
# Generate printable ASCII characters 33-126, excluding the space character.
# The supplied character images are ready to use, so regeneration is optional.
if REGENERATE_CHARACTERS:
    size = 20
    for i in range(33, 127):
        ascii_images(i, size)
else:
    print("Using the supplied character images.")


### Optional: compare two images
Provide `data/A.png` and `data/B.png` before running this cell.

In [ ]:
# Compare two example images using mean squared pixel error.
# Both images must have the same dimensions.
from sklearn.metrics import mean_squared_error
import numpy as np
from PIL import Image

img1 = np.array(Image.open(DATA_DIR / "A.png").convert("L"))
img2 = np.array(Image.open(DATA_DIR / "B.png").convert("L"))

mse = mean_squared_error(img1.ravel(), img2.ravel())
print(mse)


## 2. Optional dataset download
Copy selected dataset images into `data/photos_no_class` for the baseline and `data/photos2` for SGD.

In [ ]:
# Download the example dataset through KaggleHub.
# Copy the required images into the input folders configured above.
import kagglehub

path = kagglehub.dataset_download("heavensky/image-dataset-for-unsupervised-clustering")

print("Path to dataset files:", path)


## 3. Baseline preprocessing

In [ ]:
# Check the baseline greyscale and padding process on one example image.
# Round the dimensions up to multiples of 20 so every tile is complete.
import os
from PIL import Image, ImageOps

img = Image.open(EXAMPLE_IMAGE)
greyscale = img.convert('L')

print(greyscale.size)
x = 0
y = 0
if (greyscale.size[0]%20 != 0):
    x = 20 - greyscale.size[0]%20
if (greyscale.size[1]%20 != 0):
    y = 20 - greyscale.size[1]%20
print(f"needs {x} more pixels on the width, and {y} more pixels on the height")

size = greyscale.size[0] + x, greyscale.size[1] + y
greyscale = ImageOps.pad(greyscale, size, color = "#fff")
greyscale.show()


In [ ]:
# Prepare the baseline dataset as greyscale images with white padding.
# Save numbered JPEGs in modified_images for the next tiling step.
import os
from PIL import Image, ImageOps

folder_path = str(DATA_DIR / "photos_no_class")
i = 0
for filename in os.listdir(folder_path):
    path = os.path.join(folder_path, filename)
    img = Image.open(path)
    greyscale = img.convert('L')

    x = 0
    y = 0
    if (greyscale.size[0]%20 != 0):
        x = 20 - greyscale.size[0]%20
    if (greyscale.size[1]%20 != 0):
        y = 20 - greyscale.size[1]%20

    size = greyscale.size[0] + x, greyscale.size[1] + y
    greyscale = ImageOps.pad(greyscale, size, color = "#fff")
    new_filename = f"{i}.jpg"
    save_folder = str(DATA_DIR / "modified_images")
    save_path = os.path.join(save_folder, new_filename)
    greyscale.save(save_path)
    i = i+1


In [ ]:
# Split each prepared baseline image into non-overlapping 20x20 tiles.
# Name each tile using its source image and position in the sequence.
import os
from PIL import Image, ImageOps

folder_path = str(DATA_DIR / "modified_images")
save_folder = str(DATA_DIR / "image_bits")

total_num_crops = 0
for filename in os.listdir(folder_path):
    path = os.path.join(folder_path, filename)
    img = Image.open(path)

    l=0
    r=20
    t=0
    b=20

    width = img.size[0]
    height = img.size[1]

    x = width//20
    y = height//20

    z = 0

    for _ in range(y):
        for _ in range(x):
            crop_area = (l,t,r,b)
            # Extract the next tile in row order.
            cropped_img = img.crop(crop_area)
            new_filename = f"{Path(filename).stem}_{z}.jpg"
            save_path = os.path.join(save_folder, new_filename)
            cropped_img.save(save_path)
            l = (l+20)
            r = (r+20)
            if(l > width or r > width):
                l=0
                r=20
            z=z+1
        t = (t+20)
        b = (b+20)

    total_num_crops = total_num_crops + z

print(total_num_crops)


## 4. Baseline labeling and random forest

In [ ]:
# Assign each baseline tile the character with the lowest pixel error.
# Normalize the image pixels and flatten each tile into a feature vector.
# This early experiment retains uint8 distance arithmetic, which can overflow.
import os
import numpy as np
from PIL import Image, ImageOps
import time

folder_path = str(DATA_DIR / "image_bits")
chars_path = str(ASSET_DIR / "ascii_images")

count = 0
for _ in os.listdir(folder_path):
    count = count + 1

images = []
labels = []
i = 0
start_time = time.time()

for filename in os.listdir(folder_path):

    path = os.path.join(folder_path, filename)
    img = Image.open(path)

    # Track the closest character while comparing the templates.
    min_error = float('inf')
    char_label = None
    for char_file in os.listdir(chars_path):

        c_path = os.path.join(chars_path, char_file)
        char_img = Image.open(c_path)

        imgA = np.array(img)
        imgB = np.array(char_img)

        err = np.sum((imgA - imgB)**2)

        num_pixels = np.prod(imgA.shape)

        mse = err / num_pixels

        if (mse < min_error):
            min_error = mse
            char_label = char_file

    # Normalize the pixels before storing the model inputs.
    image = np.asarray(img, dtype = np.float32) / 255.0
    image = image.reshape(-1)

    images.append(image)
    labels.append(char_label)
    i = i +1

    print_skip = 5000
    # Report progress at a fixed interval.
    if (i % print_skip == 0):
        print(f"{(i/count)*100}% complete")
        end_time = time.time()
        time_gap = end_time - start_time
        print(f"that took {time_gap} seconds")
        print(f"estimated time to completion is {time_gap * ((count - i)/print_skip)}\n")
        start_time = time.time()

x = np.vstack(images)
y = np.vstack(labels)


In [ ]:
# Check the feature dimensions and turn the labels into a one-dimensional array.
# Scikit-learn expects one label per row of features.
print(x.shape)
print(y.shape)
y = np.ravel(y)
print(y.shape)


In [ ]:
# Train and evaluate the random forest baseline with an 80/20 tile split.
# Stratification keeps the class proportions similar in both sets.
# Tiles from the same source image can occur in both sets.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import time

start = time.time()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state=42, stratify= y)

randomforest = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)
randomforest.fit(x_train, y_train)

print(f"training took {time.time() - start} seconds")
start = time.time()
pred = randomforest.predict(x_test)
print(x_test.shape)
print(f"prediction took {time.time() - start} seconds")
print(classification_report(y_test, pred))


In [ ]:
# Save the random forest baseline for later loading or comparison.
# This is a separate model from the SGD classifier used by the webpage.
import joblib
joblib.dump(randomforest, str(MODEL_DIR / "random_forest_model.joblib"))


In [ ]:
# Load the 20x20 character images into memory for baseline reconstruction.
# Copies remain available after their source files are closed.
chars_path = str(ASSET_DIR / "ascii_images")

char_cache = {}
for char_file in os.listdir(chars_path):
    p = os.path.join(chars_path, char_file)
    with Image.open(p) as im:
        char_cache[char_file] = im.copy()


In [ ]:
# Rebuild a baseline image from the predicted character filenames.
# Paste the character tiles from left to right, then move down one row.
def image_maker(width, height, preds):

    l=0
    r=20
    t=0
    b=20

    index = 0

    output = Image.new("L", (width, height), color = 255)

    for _ in range(height//20):
        for _ in range(width//20):
            char_img = char_cache[preds[index]]
            # Place the selected character at the matching tile position.
            output.paste(char_img, (l, t))

            index = index + 1

            l = (l+20)
            r = (r+20)
            if(l > width or r > width):
                l=0
                r=20
        t = (t+20)
        b = (b+20)
    return output


In [ ]:
# Run the baseline model on the first prepared image in the folder.
# Normalize each 20x20 tile, predict its character, and display the result.
i = 0
folder_path = str(DATA_DIR / "modified_images")

img
output = 0
for filename in os.listdir(folder_path):
    if (i == 1):
        break
    path = os.path.join(folder_path, filename)
    img = Image.open(path)
    i = i+1
    l=0

    r=20

    t=0

    b=20

    width = img.size[0]
    height = img.size[1]
    x = width//20
    y = height//20

    a = 0
    batch_img = []
    start = time.time()
    for _ in range(y):
        for _ in range(x):

            crop_area = (l,t,r,b)
            # Extract the next tile in row order.
            cropped_img = img.crop(crop_area)

            img_arr = np.asarray(cropped_img, dtype=np.float32) / 255.0
            img_arr_flt = img_arr.flatten()
            batch_img.append(img_arr_flt)

            l = (l+20)
            r = (r+20)
            if(l > width or r > width):
                l=0

                r=20

        t = (t+20)
        b = (b+20)
    batch_arr = np.array(batch_img)
    batch_pred = randomforest.predict(batch_arr)

    output_img = image_maker(width, height, batch_pred)
    output_img.show()
    print(time.time() - start)


## 5. Smaller glyphs and edge-based SGD training data

In [ ]:
# Optionally reduce the 20x20 character images to 10x10 for the SGD experiment.
# Use LANCZOS resizing followed by sharpening, as in the original workflow.
# Leave regeneration disabled to keep the supplied character images unchanged.
import os
from PIL import Image, ImageFilter

if REGENERATE_CHARACTERS:
    
    src = str(ASSET_DIR / "ascii_images")
    dst = str(ASSET_DIR / "ascii_images_shrunk")
    for fn in os.listdir(src):
        path = os.path.join(src, fn)
        try:
            with Image.open(path) as img:
                w, h = img.size
                img_half = img.resize((w // 2, h // 2), Image.LANCZOS)
                img_half = img_half.filter(ImageFilter.SHARPEN)
                img_half.save(os.path.join(dst, fn))
        except OSError as error:
            print(f"Could not process {fn}: {error}")
else:
    print("Using the supplied character images.")


In [ ]:
# Prepare the SGD training images using greyscale and edge detection.
# Pad to multiples of 10 with black fill, then save the processed JPEGs.
# The final inference function below does not apply this edge filter.
import os
from PIL import Image, ImageOps

char_size = 10

folder_path = str(DATA_DIR / "photos2")
i = 0
for filename in os.listdir(folder_path):
    path = os.path.join(folder_path, filename)
    img = Image.open(path)
    greyscale = img.convert('L')
    greyscale = greyscale.filter(ImageFilter.FIND_EDGES)

    x = 0
    y = 0
    if (greyscale.size[0]%char_size != 0):
        x = char_size - greyscale.size[0]%char_size
    if (greyscale.size[1]%char_size != 0):
        y = char_size - greyscale.size[1]%char_size

    size = greyscale.size[0] + x, greyscale.size[1] + y
    greyscale = ImageOps.pad(greyscale, size, color = "#000")
    new_filename = f"{i}.jpg"
    save_folder = str(DATA_DIR / "modified_images2")
    save_path = os.path.join(save_folder, new_filename)
    greyscale.save(save_path)
    i = i+1


In [ ]:
# Split the edge-filtered training images into non-overlapping 10x10 tiles.
# Save the tiles individually and report progress every 100 source images.
import os
from PIL import Image, ImageOps

num_files = len(list((DATA_DIR / "modified_images2").iterdir()))

char_size = 10
start_total = time.time()
start_time = start_total

folder_path = str(DATA_DIR / "modified_images2")
save_folder = str(DATA_DIR / "image_bits2")

total_num_crops = 0
i = 0
for filename in os.listdir(folder_path):

    path = os.path.join(folder_path, filename)
    img = Image.open(path)

    l=0
    r=char_size
    t=0
    b=char_size

    width = img.size[0]
    height = img.size[1]

    x = width//char_size
    y = height//char_size

    z = 0

    for _ in range(y):
        for _ in range(x):
            crop_area = (l,t,r,b)
            # Extract the next tile in row order.
            cropped_img = img.crop(crop_area)
            new_filename = f"{Path(filename).stem}_{z}.jpg"
            save_path = os.path.join(save_folder, new_filename)
            cropped_img.save(save_path)
            l = (l+char_size)
            r = (r+char_size)
            if(l > width or r > width):
                l=0
                r=char_size
            z=z+1
        t = (t+char_size)
        b = (b+char_size)

    total_num_crops = total_num_crops + z

    i = i + 1

    print_skip = 100
    # Report progress at a fixed interval.
    if (i % print_skip == 0):
        print(f"{(i/num_files)*100}% complete")
        end_time = time.time()
        time_gap = end_time - start_time
        print(f"that took {time_gap} seconds")
        print(f"estimated time to completion is {time_gap * ((num_files - i)/print_skip)}\n")
        start_time = time.time()

print(total_num_crops)
print(time.time()-start_total)


In [ ]:
# Count the generated SGD training tiles and report the time taken.
image_bits_folder = str(DATA_DIR / "image_bits2")
count = 0
start = time.time()
for file in os.listdir(image_bits_folder):
    count = count + 1
print(count)
print(f"that took {time.time() - start} seconds")


## 6. Feature chunks and nearest-glyph labels

In [ ]:
# Load the supplied 10x10 character templates as normalized feature vectors.
# Keep the filenames in the same order as the vectors for label lookup.
chars_path = str(ASSET_DIR / "ascii_images_shrunk")

char_cache = []
char_files = []
for char_file in os.listdir(chars_path):
    p = os.path.join(chars_path, char_file)
    char_files.append(char_file)
    with Image.open(p) as img:
        img_arr = np.asarray(img, dtype = np.float16) / 255.0
        img_arr = img_arr.flatten()
        char_cache.append(img_arr)

char_cache = np.array(char_cache)
print(char_cache.shape)
print(char_files)


In [ ]:
# Inspect the normalized character-template array before assigning labels.
print(char_cache)


In [ ]:
# Label each training tile using its closest character template by mean squared error.
# Process the template comparisons as arrays instead of separate image loops.
# Save features and labels in chunks of 100,000 tiles to limit memory use.
# Save the remaining tiles after the loop, even when the last chunk is incomplete.
import os
import numpy as np
from PIL import Image, ImageOps
import time

folder_path = str(DATA_DIR / "image_bits2")
chars_path = str(ASSET_DIR / "ascii_images_shrunk")

save_path = str(DATA_DIR / "saved_arrays")

count = len(os.listdir(folder_path))

images = []
labels = []

chunk_idx = 0

i = 0
start_time = time.time()

for filename in os.listdir(folder_path):

    path = os.path.join(folder_path, filename)
    with Image.open(path) as img:

        min_error = float('inf')
        char_label = None

        image = np.asarray(img, dtype = np.float16) / 255.0
        image = image.reshape(-1)

        # Compare this tile with every character template at once.
        diffs = char_cache - image
        mse_all = np.mean(diffs * diffs, axis=1)
        best_idx = int(np.argmin(mse_all))
        char_label = char_files[best_idx]
        images.append(image)
        labels.append(char_label)

    i = i +1

    print_skip = 100000
    # Report progress at a fixed interval.
    if (i % print_skip == 0):
        print(f"{(i/count)*100}% complete")
        end_time = time.time()
        time_gap = end_time - start_time
        print(f"that took {time_gap} seconds")
        print(f"estimated time to completion is {time_gap * ((count - i)/print_skip)}\n")
        start_time = time.time()

        # Save this feature chunk, then release its lists from memory.
        x = np.vstack(images)
        y = np.array(labels, dtype=object)

        x_path = os.path.join(save_path, f"X_{chunk_idx:04d}.npy")
        y_path = os.path.join(save_path, f"Y_{chunk_idx:04d}.npy")

        np.save(x_path, x)
        np.save(y_path, y)

        images = []
        labels = []
        chunk_idx = chunk_idx + 1

# Save the final incomplete feature chunk.
if images:
    np.save(os.path.join(save_path, f"X_{chunk_idx:04d}.npy"), np.vstack(images))
    np.save(os.path.join(save_path, f"Y_{chunk_idx:04d}.npy"), np.array(labels, dtype=object))


## 7. Incremental SGD training and evaluation
Run only after generating feature chunks. Training overwrites the supplied model when the export cell is run.

In [ ]:
# Train the SGD classifier incrementally from the saved feature chunks.
# Shuffle the chunk pairs and reserve 20% for testing.
# Supply all possible classes on the first partial_fit call.
# Evaluate character-label agreement on the held-out chunks.
# This split does not guarantee separation between original source images.
import os
import numpy as np
import time
from sklearn.linear_model import SGDClassifier

saved_path = str(DATA_DIR / "saved_arrays")
chars_path = str(ASSET_DIR / "ascii_images_shrunk")

char_files = []
for f in os.listdir(chars_path):
    if (f.lower().endswith(".png")):
        char_files.append(f)

char_files = sorted(char_files, key=lambda x: int(os.path.splitext(x)[0]))
classes = np.array(char_files, dtype=object)

X_files = [f for f in os.listdir(saved_path) if (f.startswith("X_") and f.endswith(".npy"))]
Y_files = [f for f in os.listdir(saved_path) if (f.startswith("Y_") and f.endswith(".npy"))]

X_files = sorted(X_files)
Y_files = sorted(Y_files)

if (len(X_files) != len(Y_files)):
    print("WARNING: number of X chunks != number of Y chunks")
    print("X chunks:", len(X_files))
    print("Y chunks:", len(Y_files))

pairs = list(zip(X_files, Y_files))

# Shuffle the feature chunks before making the train/test split.
rng = np.random.default_rng(0)
rng.shuffle(pairs)

n_total = len(pairs)
n_train = int(n_total * 0.8)

train_pairs = pairs[:n_train]
test_pairs  = pairs[n_train:]

print("total chunks:", n_total)
print("train chunks:", len(train_pairs))
print("test chunks:", len(test_pairs))

# Use logistic loss with L2 regularization for incremental training.
clf = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-5,
    max_iter=1,
    tol=None,
    random_state=0
)

start_time = time.time()

k = 0
for (xf, yf) in train_pairs:
    X = np.load(os.path.join(saved_path, xf))
    y = np.load(os.path.join(saved_path, yf), allow_pickle=True)

    X = X.astype(np.float32, copy=False)

    # The first update needs the full list of possible character labels.
    if (k == 0):
        clf.partial_fit(X, y, classes=classes)
    else:
        clf.partial_fit(X, y)

    k = k + 1

    print_skip = 10
    if (k % print_skip == 0):
        time_gap = time.time() - start_time
        print(f"trained {k}/{len(train_pairs)} train chunks")
        print(f"that took {time_gap} seconds\n")
        start_time = time.time()

print("training done")

# Accumulate correct predictions across all held-out chunks.
correct = 0
total = 0

start_time = time.time()
k = 0
for (xf, yf) in test_pairs:
    X = np.load(os.path.join(saved_path, xf))
    y = np.load(os.path.join(saved_path, yf), allow_pickle=True)

    X = X.astype(np.float32, copy=False)

    pred = clf.predict(X)

    correct = correct + int(np.sum(pred == y))
    total = total + int(len(y))

    k = k + 1

    print_skip = 10
    if (k % print_skip == 0):
        acc = correct / total if total > 0 else 0
        time_gap = time.time() - start_time
        print(f"tested {k}/{len(test_pairs)} test chunks")
        print(f"running accuracy: {acc}")
        print(f"that took {time_gap} seconds\n")
        start_time = time.time()

final_acc = correct / total if total > 0 else 0
print("final test accuracy:", final_acc)


In [ ]:
# Export the trained SGD classifier to the configured model folder.
# Running this cell replaces the existing model at that location.
from joblib import dump

model_path = str(MODEL_DIR / "sgd_ascii_model.joblib")
dump(clf, model_path)

print("model saved to:", model_path)


## 8. Load the trained model and reconstruct images

In [ ]:
# Load the saved SGD classifier without repeating training.
from joblib import load

model_path = str(MODEL_DIR / "sgd_ascii_model.joblib")

clf = load(model_path)
print("model loaded")


In [ ]:
# Load the original 10x10 character PNGs into a filename-to-image cache.
# The same supplied PNGs are used by the standalone converter.
from PIL import Image
chars_path = str(ASSET_DIR / "ascii_images_shrunk")

char_cache = {}
for char_file in os.listdir(chars_path):
    p = os.path.join(chars_path, char_file)
    with Image.open(p) as im:
        char_cache[char_file] = im.copy()


In [ ]:
# Reconstruct an image from predicted character filenames and a chosen tile size.
# Paste each character into the position occupied by its input tile.
def image_maker(width, height, preds, size):

    l=0
    r=size
    t=0
    b=size

    index = 0

    output = Image.new("L", (width, height), color = 255)

    for _ in range(height//size):
        for _ in range(width//size):
            char_img = char_cache[preds[index]]
            # Place the selected character at the matching tile position.
            output.paste(char_img, (l, t))

            index = index + 1

            l = (l+size)
            r = (r+size)
            if(l > width or r > width):
                l=0
                r=size
        t = (t+size)
        b = (b+size)
    return output


### Intermediate inference experiment
This cell expects preprocessed grayscale images whose dimensions are multiples of 10.

In [ ]:
# Test the SGD model on the first image in the example folder.
# This intermediate experiment expects preprocessed greyscale images
# with dimensions divisible by 10. The final function handles preparation.
from PIL import ImageFilter
import time

i = 0
folder_path = str(DATA_DIR / "test_images")

size = 10

output = 0
for filename in os.listdir(folder_path):
    if (i == 1):
        break
    path = os.path.join(folder_path, filename)
    img = Image.open(path)
    i = i+1
    l=0

    r=size

    t=0

    b=size

    width = img.size[0]
    height = img.size[1]
    x = width//size
    y = height//size

    a = 0
    batch_img = []
    start = time.time()
    for _ in range(y):
        for _ in range(x):
            crop_area = (l,t,r,b)
            # Extract the next tile in row order.
            cropped_img = img.crop(crop_area)
            img_arr = np.asarray(cropped_img, dtype=np.float32) / 255.0
            img_arr_flt = img_arr.flatten()
            batch_img.append(img_arr_flt)

            l = (l+size)
            r = (r+size)
            if(l > width or r > width):
                l=0

                r=size

        t = (t+size)
        b = (b+size)
    batch_arr = np.array(batch_img)
    print(batch_arr.shape)
    batch_pred = clf.predict(batch_arr)

    output_img = image_maker(width, height, batch_pred, size)
    output_img.show()
    print(time.time() - start)


## 9. Final inference pipeline
This grayscale/padding pipeline is used by `converter.py`.

In [ ]:
# Prepare an input image for the final conversion pipeline.
# Convert to greyscale and pad with black to reach complete tile dimensions.
# ImageOps.pad can resize slightly while fitting the requested dimensions.
import os
from PIL import Image, ImageOps

def image_modifier(img, char_size):
    greyscale = img.convert('L')
    x = 0
    y = 0
    if (greyscale.size[0]%char_size != 0):
        x = char_size - greyscale.size[0]%char_size
    if (greyscale.size[1]%char_size != 0):
        y = char_size - greyscale.size[1]%char_size

    size = greyscale.size[0] + x, greyscale.size[1] + y
    greyscale = ImageOps.pad(greyscale, size, color = "#000")
    return greyscale


In [ ]:
# Load the SGD model used by the final image-to-ASCII function.
from joblib import load

model_path = str(MODEL_DIR / "sgd_ascii_model.joblib")

model = load(model_path)
print("model loaded")


In [ ]:
# Convert one image using the final notebook pipeline.
# Prepare the image, flatten normalized tiles, predict character filenames,
# and combine the matching character images into the finished output.
import os
import numpy as np
from PIL import Image

def image_to_ascii(folder, filename, tile_size, model):

    path = os.path.join(folder, filename)
    orig = Image.open(path)

    orig.show()

    img = image_modifier(orig, tile_size)

    img.show()

    l=0

    r=tile_size

    t=0

    b=tile_size

    width = img.size[0]
    height = img.size[1]
    x = width//tile_size
    y = height//tile_size

    batch_img = []
    for _ in range(y):
        for _ in range(x):
            crop_area = (l,t,r,b)
            # Extract the next tile in row order.
            cropped_img = img.crop(crop_area)
            img_arr = np.asarray(cropped_img, dtype=np.float32) / 255.0
            img_arr_flt = img_arr.flatten()
            batch_img.append(img_arr_flt)

            l = (l+tile_size)
            r = (r+tile_size)
            if(l > width or r > width):

                l=0

                r=tile_size

        t = (t+tile_size)
        b = (b+tile_size)
    batch_arr = np.array(batch_img)
    # Predict all tile labels before reconstructing the output image.
    batch_pred = model.predict(batch_arr)

    output_img = image_maker(width, height, batch_pred, tile_size)
    output_img.show()


In [ ]:
# Run the final converter on the example image configured at the top.
# Keep the tile size at 10 to match the saved model's 100 input features.
folder = str(EXAMPLE_IMAGE.parent)
filename = EXAMPLE_IMAGE.name
tile_size = 10

SGD_classifier = model

image_to_ascii(folder, filename, tile_size, SGD_classifier)
